# 🧬 SpeculativeConfig(mtp) の実構築

target_parallel_config 付きで draft config の解決を確認。


In [ ]:
import subprocess
import sys

print("=== pip install vllm==0.32.0 ===")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input",
                    "vllm==0.32.0"], capture_output=True, text=True, timeout=5400)
tail = ((r.stdout or "") + (r.stderr or "")).strip().splitlines()
print("   rc:", r.returncode, "|", " / ".join(tail[-3:])[:300])

def find_root():
    import glob
    slug = "qwen38-27b-uncensored-hauhaucs-awq-mtp"
    ordered = []
    for pat in ("/kaggle/input/*/config.json", "/kaggle/input/*/*/config.json",
                "/kaggle/input/*/*/*/config.json"):
        ordered += glob.glob(pat)
    cands = ([c for c in ordered if slug in c]
             or [c for c in ordered if "qwen" in c.lower()]
             or ordered)
    print("config.json 候補:", cands[:3])
    if not cands:
        print("/kaggle/input 直下:", glob.glob("/kaggle/input/*")[:10])
        raise SystemExit("dataset が見つからない")
    return cands[0].rsplit("/", 1)[0]

ROOT = find_root()

CASE = r"""
import json
import sys

ROOT = sys.argv[1]
from vllm.config import ModelConfig, SpeculativeConfig
from vllm.config.parallel import ParallelConfig


def show(tag, fn):
    try:
        v = fn()
        print(f"[OK] {tag}")
        return v
    except Exception as e:
        print(f"[NG] {tag}: {type(e).__name__}: {str(e)[:500]}")
        return None


target = ModelConfig(model=ROOT, dtype="half", max_model_len=8192,
                     trust_remote_code=True)
print("target: quant=", target.quantization, "arch=", target.hf_config.architectures)
tc = target.hf_config.text_config
print("text_config: num_hidden_layers=", tc.num_hidden_layers,
      "mtp_num_hidden_layers=", getattr(tc, "mtp_num_hidden_layers", None),
      "num_kv_heads=", getattr(tc, "num_key_value_heads", None),
      "head_dim=", getattr(tc, "head_dim", None),
      "layer_types=", (getattr(tc, "layer_types", None) or [])[:6])

par = show("ParallelConfig(tp=2)", lambda: ParallelConfig(tensor_parallel_size=2))
if not par:
    raise SystemExit("ParallelConfig が作れない")

for kw in (dict(num_speculative_tokens=1),
           dict(num_speculative_tokens=2),
           dict(num_speculative_tokens=3),
           dict(num_speculative_tokens=4),
           dict(),
           dict(num_speculative_tokens=1, model=ROOT),
           dict(num_speculative_tokens=2, draft_tensor_parallel_size=2)):
    tag = json.dumps(kw)
    sp = show(f"Spec {tag}", lambda kw=kw: SpeculativeConfig(
        method="mtp", target_model_config=target,
        target_parallel_config=par, **kw))
    if not sp:
        continue
    dm = sp.draft_model_config
    print(f"    n={sp.num_speculative_tokens} method={sp.method} "
          f"draft_tp={getattr(sp, 'draft_tensor_parallel_size', None)}")
    print(f"    draft.model={dm.model!r}")
    print(f"    draft.arch={dm.hf_config.architectures} "
          f"model_type={dm.hf_config.model_type}")
    print(f"    draft.n_predict={getattr(dm.hf_config, 'n_predict', None)} "
          f"mtp_num_hidden_layers={getattr(dm.hf_config, 'mtp_num_hidden_layers', None)}")
    print(f"    draft.quant={dm.quantization!r} dtype={dm.dtype} "
          f"max_model_len={dm.max_model_len}")
    print(f"    draft_parallel.tp={getattr(sp.draft_parallel_config, 'tensor_parallel_size', None)}")

print("\\n=== draft が target と同じディレクトリを指すか（重要）===")
sp = SpeculativeConfig(method="mtp", num_speculative_tokens=1,
                       target_model_config=target, target_parallel_config=par)
print("  target.model == draft.model:", target.model == sp.draft_model_config.model)
print("  値:", target.model, "|", sp.draft_model_config.model)

print("\\n=== 量子化明示との組み合わせ ===")
tct = ModelConfig(model=ROOT, dtype="half", max_model_len=8192,
                  quantization="compressed-tensors", trust_remote_code=True)
sp2 = show("Spec + ct", lambda: SpeculativeConfig(
    method="mtp", num_speculative_tokens=1, target_model_config=tct,
    target_parallel_config=par))
if sp2:
    print("  draft.quant:", sp2.draft_model_config.quantization)

print("\\n=== method の正規化 ===")
for m in ("mtp", "qwen3_5_mtp", "qwen3_next_mtp", "MTP_model"):
    sp3 = show(f"method={m}", lambda m=m: SpeculativeConfig(
        method=m, num_speculative_tokens=1, target_model_config=target,
        target_parallel_config=par))
    if sp3:
        print("    -> method:", sp3.method)
"""

import os
import subprocess
import sys
import tempfile

case = os.path.join(tempfile.gettempdir(), "speccase.py")
open(case, "w").write(CASE)
r = subprocess.run([sys.executable, case, ROOT], capture_output=True, text=True,
                   timeout=3600)
print((r.stdout or "")[-14000:])
if r.returncode != 0:
    print("=== stderr ===")
    print((r.stderr or "")[-4000:])
